# Grid de pesos do Composable CFG (s_id × s_clip × s_attr)

Gera via **DDIM** a partir de uma foto (sem editar nenhum atributo — o vetor
usado é o da própria foto), variando os pesos de guidance de **1 a 10**.

Um grid completo 10×10×10 seriam 1000 gerações (~horas). Por padrão este
notebook faz o corte informativo: **3 varreduras 1-D** (cada peso varia de 1 a 10
com os outros dois fixos no baseline) = 30 gerações. A célula final faz um
grid 2-D opcional (`RUN_2D = True`).

Todas as células do grid usam a **mesma seed** → mesmo ruído inicial z_T;
como o DDIM é determinístico (eta=0), a ÚNICA coisa que muda entre as
imagens são os pesos.


In [ ]:
# ============ CONFIG ============
PHOTO           = "nova.jpeg"
CKPT            = "models/LDM_CFGComp_split_paired/ckpt_best.pt"   # 3 pesos -> checkpoint clip_arcface_split
VAE_CKPT        = "vae/vae_epoch_62.pt"
CLASSIFIER_CKPT = "models/attribute_classifier/ckpt_best.pt"        # None p/ usar ORIG_ATTRS
ORIG_ATTRS      = ["Young", "Male", "Black_Hair", "Mustache"]       # usado só se CLASSIFIER_CKPT=None

WEIGHTS    = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
BASE       = dict(s_id=3.0, s_clip=3.0, s_attr=5.0)  # valores fixos enquanto um peso varia

DDIM_STEPS = 50
SEED       = 0
DEVICE     = "cuda"
SAVE_DIR   = "results/grid_weights"

RUN_2D     = False              # grid 2-D s_id x s_clip (celula final)
WEIGHTS_2D = [1, 3, 5, 7, 9]    # eixos do grid 2-D (5x5 = 25 geracoes)


In [ ]:
# ============ PIPELINE + FOTO + ATRIBUTOS ============
import os
from types import SimpleNamespace

import torch
import matplotlib.pyplot as plt
from torchvision.utils import save_image

from utils.edit_common import (
    EditPipeline, prepare_photo, resolve_original_attrs, CELEBA_ATTRS,
)
from utils.composable_cfg_sampling import sample_split_ddim, sample_composable_ddim

os.makedirs(SAVE_DIR, exist_ok=True)

ref_img = prepare_photo(PHOTO, DEVICE)          # [1,3,256,256] em [-1,1]
pipeline = EditPipeline(ckpt_path=CKPT, vae_ckpt=VAE_CKPT, device=DEVICE)
print(f"encoder: {pipeline.encoder_type}")

if pipeline.encoder_type != "clip_arcface_split":
    print("AVISO: checkpoint sem ramo CLIP separado — s_clip nao se aplica; "
          "a varredura de s_clip sera pulada.")

# atributos da propria foto, SEM edicao
_args = SimpleNamespace(orig_attrs=None if CLASSIFIER_CKPT else ORIG_ATTRS,
                        classifier_ckpt=CLASSIFIER_CKPT)
attrs_vec = resolve_original_attrs(_args, PHOTO, DEVICE)
print("Atributos usados:", [CELEBA_ATTRS[i] for i in range(40) if attrs_vec[i] == 1.0])


In [ ]:
# ============ TOKENS (calculados uma vez) ============
with torch.no_grad():
    attr_tok = pipeline.attribute_embedder(attrs_vec.view(1, -1).to(DEVICE))
    if pipeline.encoder_type == "clip_arcface_split":
        clip_tok, id_tok = pipeline.image_encoder(ref_img=ref_img, return_separate=True)
    else:
        id_tok, clip_tok = pipeline.image_encoder(ref_img=ref_img), None

ref_vis = ((ref_img.squeeze(0).clamp(-1, 1) + 1) / 2).cpu()
plt.figure(figsize=(3, 3)); plt.imshow(ref_vis.permute(1, 2, 0)); plt.axis("off")
plt.title("entrada alinhada"); plt.show()


In [ ]:
# ============ GERACAO DE UMA CELULA DO GRID ============
@torch.no_grad()
def gen(s_id, s_clip, s_attr):
    """Gera 1 imagem com os pesos dados. Seed fixa -> mesmo z_T em todas
    as celulas; com DDIM (eta=0) so os pesos mudam o resultado."""
    torch.manual_seed(SEED)
    if pipeline.encoder_type == "clip_arcface_split":
        z = sample_split_ddim(
            unet=pipeline.unet, diffusion=pipeline.diffusion,
            id_tokens=id_tok, clip_tokens=clip_tok, attr_tokens=attr_tok,
            n=1, channels=4, device=DEVICE,
            s_id=s_id, s_clip=s_clip, s_attr=s_attr, ddim_steps=DDIM_STEPS,
        )
    else:
        z = sample_composable_ddim(
            unet=pipeline.unet, diffusion=pipeline.diffusion,
            id_tokens=id_tok, attr_tokens=attr_tok,
            n=1, channels=4, device=DEVICE,
            s_id=s_id, s_attr=s_attr, ddim_steps=DDIM_STEPS,
        )
    img = pipeline.decode(z)
    return ((img.squeeze(0).clamp(-1, 1) + 1) / 2).cpu()


def sweep(axis, values):
    """Varre um peso mantendo os outros no BASE. Mostra e salva a linha."""
    imgs = []
    for v in values:
        kw = dict(BASE); kw[axis] = float(v)
        print(f"{axis}={v}  (fixos: {({k: x for k, x in kw.items() if k != axis})})")
        imgs.append(gen(**kw))

    cols = len(values) + 1
    fig, axes = plt.subplots(1, cols, figsize=(1.9 * cols, 2.3))
    axes[0].imshow(ref_vis.permute(1, 2, 0)); axes[0].set_title("entrada", fontsize=9)
    for ax, v, im in zip(axes[1:], values, imgs):
        ax.imshow(im.permute(1, 2, 0)); ax.set_title(f"{axis}={v}", fontsize=9)
    for ax in axes: ax.axis("off")
    fixos = "  ".join(f"{k}={v}" for k, v in BASE.items() if k != axis)
    fig.suptitle(f"varredura de {axis}   (fixos: {fixos})", fontsize=11)
    plt.tight_layout(); plt.show()

    out = os.path.join(SAVE_DIR, f"sweep_{axis}_seed{SEED}.png")
    save_image(torch.stack([ref_vis] + imgs), out, nrow=cols)
    print(f"salvo: {out}")
    return imgs


In [ ]:
# ============ VARREDURA 1: s_id (identidade / ArcFace) ============
imgs_id = sweep("s_id", WEIGHTS)


In [ ]:
# ============ VARREDURA 2: s_clip (aparencia global / CLIP) ============
if pipeline.encoder_type == "clip_arcface_split":
    imgs_clip = sweep("s_clip", WEIGHTS)
else:
    print("pulado: checkpoint sem ramo CLIP separado.")


In [ ]:
# ============ VARREDURA 3: s_attr (atributos) ============
imgs_attr = sweep("s_attr", WEIGHTS)


## Grid 2-D opcional (s_id × s_clip, s_attr fixo)

Depois de olhar as varreduras acima, ligue `RUN_2D = True` na config e ajuste
`WEIGHTS_2D` para a regiao que pareceu promissora. Linhas = `s_id`,
colunas = `s_clip`, `s_attr` fixo no BASE.


In [ ]:
if RUN_2D:
    if pipeline.encoder_type != "clip_arcface_split":
        raise RuntimeError("grid 2-D s_id x s_clip requer checkpoint clip_arcface_split")
    rows = []
    for si in WEIGHTS_2D:
        for sc in WEIGHTS_2D:
            print(f"s_id={si}  s_clip={sc}  s_attr={BASE['s_attr']}")
            rows.append(gen(s_id=float(si), s_clip=float(sc), s_attr=BASE["s_attr"]))

    n = len(WEIGHTS_2D)
    fig, axes = plt.subplots(n, n, figsize=(1.9 * n, 2.0 * n))
    for i, si in enumerate(WEIGHTS_2D):
        for j, sc in enumerate(WEIGHTS_2D):
            ax = axes[i][j]
            ax.imshow(rows[i * n + j].permute(1, 2, 0)); ax.axis("off")
            if i == 0: ax.set_title(f"s_clip={sc}", fontsize=9)
            if j == 0: ax.text(-0.08, 0.5, f"s_id={si}", fontsize=9, rotation=90,
                               va="center", ha="center", transform=ax.transAxes)
    fig.suptitle(f"s_id x s_clip   (s_attr={BASE['s_attr']}, seed={SEED})", fontsize=11)
    plt.tight_layout(); plt.show()

    out = os.path.join(SAVE_DIR, f"grid2d_sid_sclip_sattr{BASE['s_attr']}_seed{SEED}.png")
    save_image(torch.stack(rows), out, nrow=n)
    print(f"salvo: {out}")
else:
    print("RUN_2D = False — nada a fazer.")
